# Notebook 12 — Load All Data into SQLite (Task 5)
**Engine :** SQLAlchemy + sqlite3  
**Output :** `bluestock_mf.db`  
**Tables  :** dim_date · dim_fund · fact_nav · fact_transactions · fact_performance · fact_aum · fact_portfolio · fact_sip_industry

---
### What this notebook does
1. Creates `bluestock_mf.db` from `schema.sql`
2. Generates `dim_date` (2022–2026, 1,826 rows) with Indian FY fields
3. Loads all 6 fact tables from cleaned CSVs
4. Computes `daily_return_pct` in `fact_nav` during load
5. Derives `below_min_threshold` flag in `fact_transactions`
6. Verifies row counts and FK integrity after load


## 1. Install & Import Libraries

In [ ]:
# Run this cell first if libraries are missing
# !pip install sqlalchemy pandas

import sqlite3
import pathlib
import os
import pandas as pd
from sqlalchemy import create_engine, text

print(f"pandas     : {pd.__version__}")
import sqlalchemy
print(f"sqlalchemy : {sqlalchemy.__version__}")
print(f"sqlite3    : {sqlite3.sqlite_version}")

## 2. Configure Paths
⚠️ **Change these paths to match your machine before running.**

In [ ]:
# ── UPDATE THESE TWO PATHS ──────────────────────────────────────────
SCHEMA_PATH  = r'schema.sql'                  # path to schema.sql
DB_PATH      = r'bluestock_mf.db'            # where to create the database
CLEANED_DIR  = r'outputs/'                   # folder with all clean_*.csv files
# ────────────────────────────────────────────────────────────────────

# Verify schema file exists
assert pathlib.Path(SCHEMA_PATH).exists(), f"schema.sql not found at: {SCHEMA_PATH}"
assert pathlib.Path(CLEANED_DIR).exists(), f"Cleaned data folder not found: {CLEANED_DIR}"

print("Paths verified.")
print(f"  Schema : {SCHEMA_PATH}")
print(f"  DB     : {DB_PATH}")
print(f"  Data   : {CLEANED_DIR}")

def csv(filename):
    """Helper — build full path to a cleaned CSV."""
    return os.path.join(CLEANED_DIR, filename)

## 3. Create Database & Run Schema
This step creates the `.db` file and runs all `CREATE TABLE` statements from `schema.sql`.

In [ ]:
# Remove existing DB so we start fresh (comment out if you want to keep existing data)
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
    print("Existing DB removed.")

# Read and execute schema
schema_sql = pathlib.Path(SCHEMA_PATH).read_text()
conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = OFF")  # OFF during bulk load — re-enabled at the end
conn.executescript(schema_sql)
conn.commit()

print("Schema executed successfully.")
print(f"Database file created: {DB_PATH}")

# List tables created
tables = conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
).fetchall()
print(f"\nTables created ({len(tables)}):")
for t in tables:
    print(f"  - {t[0]}")

## 4. Load dim_date
Generates one row per calendar day from 2022-01-01 to 2026-12-31.  
Includes Indian Financial Year fields (April → March).

In [ ]:
def financial_year(d):
    if d.month >= 4:
        return f"FY{d.year}-{str(d.year+1)[2:]}"
    return f"FY{d.year-1}-{str(d.year)[2:]}"

def financial_quarter(d):
    m = d.month
    if   m in [4,5,6]:    q, fy = 1, d.year
    elif m in [7,8,9]:    q, fy = 2, d.year
    elif m in [10,11,12]: q, fy = 3, d.year
    else:                  q, fy = 4, d.year - 1
    return f"Q{q}FY{str(fy+1)[2:]}"

def financial_month_num(d):
    return ((d.month - 4) % 12) + 1

QTR_END_MONTHS = {3, 6, 9, 12}
date_range     = pd.date_range('2022-01-01', '2026-12-31', freq='D')

rows = []
for d in date_range:
    rows.append({
        'date_id'            : d.strftime('%Y-%m-%d'),
        'year'               : d.year,
        'quarter'            : (d.month - 1) // 3 + 1,
        'month'              : d.month,
        'month_name'         : d.strftime('%B'),
        'week_of_year'       : int(d.strftime('%W')),   # 0-53
        'day_of_month'       : d.day,
        'day_of_week'        : d.weekday(),              # 0=Monday
        'day_name'           : d.strftime('%A'),
        'is_weekday'         : int(d.weekday() < 5),
        'is_month_end'       : int(d == d + pd.offsets.MonthEnd(0)),
        'is_quarter_end'     : int(d == d + pd.offsets.MonthEnd(0) and d.month in QTR_END_MONTHS),
        'is_year_end'        : int(d.month == 12 and d.day == 31),
        'financial_year'     : financial_year(d),
        'financial_quarter'  : financial_quarter(d),
        'financial_month_num': financial_month_num(d),
    })

dim_date = pd.DataFrame(rows)
dim_date.to_sql('dim_date', conn, if_exists='append', index=False)

print(f"dim_date loaded: {len(dim_date):,} rows")
print(f"Date range     : {dim_date['date_id'].min()}  →  {dim_date['date_id'].max()}")
dim_date.head(3)

## 5. Load dim_fund

In [ ]:
fund_cols = [
    'amfi_code','fund_house','scheme_name','category','sub_category',
    'plan','sebi_category_code','launch_date','fund_manager','benchmark',
    'expense_ratio_pct','exit_load_pct','min_sip_amount',
    'min_lumpsum_amount','risk_category'
]

df_fund = pd.read_csv(csv('clean_01_fund_master.csv'))[fund_cols]
df_fund.to_sql('dim_fund', conn, if_exists='append', index=False)

print(f"dim_fund loaded: {len(df_fund):,} rows")
print(f"Fund houses    : {df_fund['fund_house'].nunique()}")
print(f"Categories     : {df_fund['category'].unique().tolist()}")
df_fund.head(3)

## 6. Load fact_nav
Computes `daily_return_pct = (nav - prev_nav) / prev_nav × 100` per fund.  
First row per fund will have `NaN` (no prior day to compare).

In [ ]:
df_nav = pd.read_csv(csv('clean_02_nav_history.csv'))
df_nav = df_nav.rename(columns={'date': 'date_id'})
df_nav = df_nav.sort_values(['amfi_code', 'date_id']).reset_index(drop=True)

# Compute daily return
df_nav['daily_return_pct'] = (
    df_nav.groupby('amfi_code')['nav']
    .pct_change() * 100
)

nav_out = df_nav[['amfi_code', 'date_id', 'nav', 'daily_return_pct']]
nav_out.to_sql('fact_nav', conn, if_exists='append', index=False, chunksize=5000)

null_returns = nav_out['daily_return_pct'].isnull().sum()
print(f"fact_nav loaded      : {len(nav_out):,} rows")
print(f"NULL daily_return    : {null_returns} (one per fund — expected)")
print(f"Avg daily return     : {nav_out['daily_return_pct'].mean():.4f}%")
nav_out.head(3)

## 7. Load fact_transactions
Derives `below_min_threshold = 1` where SIP amount < min_sip_amount or Lumpsum < min_lumpsum_amount.

In [ ]:
tx = pd.read_csv(csv('clean_08_investor_transactions.csv'))
tx = tx.rename(columns={'transaction_date': 'date_id'})

# Join thresholds from fund master
thresholds = df_fund[['amfi_code', 'min_sip_amount', 'min_lumpsum_amount']]
tx = tx.merge(thresholds, on='amfi_code', how='left')

# Derive flag
tx['below_min_threshold'] = 0
tx.loc[
    (tx['transaction_type'] == 'SIP') & (tx['amount_inr'] < tx['min_sip_amount']),
    'below_min_threshold'
] = 1
tx.loc[
    (tx['transaction_type'] == 'Lumpsum') & (tx['amount_inr'] < tx['min_lumpsum_amount']),
    'below_min_threshold'
] = 1

tx_cols = [
    'investor_id', 'amfi_code', 'date_id', 'transaction_type', 'amount_inr',
    'payment_mode', 'state', 'city', 'city_tier', 'age_group', 'gender',
    'annual_income_lakh', 'kyc_status', 'below_min_threshold'
]
tx[tx_cols].to_sql('fact_transactions', conn, if_exists='append', index=False, chunksize=5000)

print(f"fact_transactions loaded : {len(tx):,} rows")
print(f"below_min_threshold = 1  : {tx['below_min_threshold'].sum()} rows (880 expected)")
print(f"Transaction type split   :")
print(tx['transaction_type'].value_counts().to_string())
tx[tx_cols].head(3)

## 8. Load fact_performance
No date column in source CSV — `as_of_date` is set to the data extraction date: `2025-05-30`.

In [ ]:
perf_cols = [
    'amfi_code', 'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct',
    'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio',
    'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct',
    'morningstar_rating', 'risk_grade'
]

df_perf = pd.read_csv(csv('clean_07_scheme_performance.csv'))[perf_cols].copy()
df_perf['as_of_date'] = '2025-05-30'

df_perf.to_sql('fact_performance', conn, if_exists='append', index=False)

print(f"fact_performance loaded: {len(df_perf):,} rows")
print(f"Sharpe ratio range     : {df_perf['sharpe_ratio'].min():.2f}  to  {df_perf['sharpe_ratio'].max():.2f}")
df_perf.head(3)

## 9. Load fact_aum

In [ ]:
df_aum = pd.read_csv(csv('clean_03_aum_by_fund_house.csv'))
df_aum = df_aum.rename(columns={'date': 'date_id'})

df_aum[['fund_house','date_id','aum_crore','aum_lakh_crore','num_schemes']]    .to_sql('fact_aum', conn, if_exists='append', index=False)

print(f"fact_aum loaded      : {len(df_aum):,} rows")
print(f"Fund houses          : {df_aum['fund_house'].unique().tolist()}")
df_aum.head(3)

## 10. Load fact_portfolio

In [ ]:
df_hold = pd.read_csv(csv('clean_09_portfolio_holdings.csv'))
df_hold = df_hold.rename(columns={'portfolio_date': 'date_id'})

port_cols = ['amfi_code','date_id','stock_symbol','stock_name',
             'sector','weight_pct','market_value_cr','current_price_inr']
df_hold[port_cols].to_sql('fact_portfolio', conn, if_exists='append', index=False)

print(f"fact_portfolio loaded: {len(df_hold):,} rows")
print(f"Unique stocks        : {df_hold['stock_symbol'].nunique()}")
print(f"Sectors              : {df_hold['sector'].nunique()}")
df_hold[port_cols].head(3)

## 11. Load fact_sip_industry

In [ ]:
df_sip = pd.read_csv(csv('clean_04_monthly_sip_inflows.csv'))
df_sip = df_sip.rename(columns={'month': 'date_id'})

sip_cols = ['date_id','sip_inflow_crore','active_sip_accounts_crore',
            'new_sip_accounts_lakh','sip_aum_lakh_crore','yoy_growth_pct']
df_sip[sip_cols].to_sql('fact_sip_industry', conn, if_exists='append', index=False)

print(f"fact_sip_industry loaded : {len(df_sip):,} rows")
print(f"yoy_growth NULL (first 12 months expected): {df_sip['yoy_growth_pct'].isnull().sum()}")
df_sip[sip_cols].head(3)

## 12. Re-enable Foreign Keys & Commit

In [ ]:
conn.execute("PRAGMA foreign_keys = ON")
conn.commit()
print("Foreign keys re-enabled. All data committed.")

## 13. Verification — Row Counts

In [ ]:
tables = [
    'dim_date', 'dim_fund',
    'fact_nav', 'fact_transactions', 'fact_performance',
    'fact_aum', 'fact_portfolio', 'fact_sip_industry'
]

print(f"{'Table':<30} {'Rows':>10}")
print("─" * 44)
total = 0
for t in tables:
    n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    total += n
    print(f"  {t:<28} {n:>10,}")
print("─" * 44)
print(f"  {'TOTAL':<28} {total:>10,}")

## 14. Verification — Sample Queries

In [ ]:
print("=== Top 5 Funds by AUM (fact_aum) ===")
q1 = conn.execute("""
    SELECT fund_house, date_id, aum_crore
    FROM fact_aum
    ORDER BY aum_crore DESC
    LIMIT 5
""").fetchall()
for row in q1:
    print(f"  {row[0]:<25}  {row[1]}  ₹{row[2]:>10,} Cr")

print()
print("=== NAV range check (fact_nav) ===")
q2 = conn.execute("""
    SELECT amfi_code,
           COUNT(*) as days,
           MIN(nav) as min_nav,
           MAX(nav) as max_nav,
           ROUND(AVG(daily_return_pct),4) as avg_daily_ret
    FROM fact_nav
    GROUP BY amfi_code
    LIMIT 5
""").fetchall()
for row in q2:
    print(f"  amfi={row[0]}  days={row[1]:,}  nav={row[2]:.2f}–{row[3]:.2f}  avg_ret={row[4]}%")

print()
print("=== Transaction breakdown (fact_transactions) ===")
q3 = conn.execute("""
    SELECT transaction_type,
           COUNT(*) as count,
           SUM(amount_inr) as total_inr
    FROM fact_transactions
    GROUP BY transaction_type
""").fetchall()
for row in q3:
    print(f"  {row[0]:<12}  count={row[1]:,}  total=₹{row[2]/1e7:.1f} Cr")

## 15. Verification — FK Integrity Check

In [ ]:
print("Checking FK integrity...")
issues = 0

# fact_nav → dim_fund
n = conn.execute("""
    SELECT COUNT(*) FROM fact_nav fn
    LEFT JOIN dim_fund df ON fn.amfi_code = df.amfi_code
    WHERE df.amfi_code IS NULL
""").fetchone()[0]
print(f"  fact_nav orphaned amfi_codes    : {n}")
issues += n

# fact_nav → dim_date
n = conn.execute("""
    SELECT COUNT(*) FROM fact_nav fn
    LEFT JOIN dim_date dd ON fn.date_id = dd.date_id
    WHERE dd.date_id IS NULL
""").fetchone()[0]
print(f"  fact_nav orphaned date_ids      : {n}")
issues += n

# fact_transactions → dim_fund
n = conn.execute("""
    SELECT COUNT(*) FROM fact_transactions ft
    LEFT JOIN dim_fund df ON ft.amfi_code = df.amfi_code
    WHERE df.amfi_code IS NULL
""").fetchone()[0]
print(f"  fact_transactions orphaned amfi : {n}")
issues += n

# fact_performance → dim_fund
n = conn.execute("""
    SELECT COUNT(*) FROM fact_performance fp
    LEFT JOIN dim_fund df ON fp.amfi_code = df.amfi_code
    WHERE df.amfi_code IS NULL
""").fetchone()[0]
print(f"  fact_performance orphaned amfi  : {n}")
issues += n

print()
if issues == 0:
    print("FK integrity check: PASSED — zero orphaned rows across all fact tables")
else:
    print(f"WARNING: {issues} orphaned rows found — investigate before proceeding")

## 16. Database File Info

In [ ]:
conn.close()

db_size = os.path.getsize(DB_PATH) / 1024 / 1024
print(f"Database file : {DB_PATH}")
print(f"File size     : {db_size:.2f} MB")
print()
print("Next step: Task 6 — Write 10 analytical SQL queries")
print("  Open DB Browser for SQLite → File → Open Database → select bluestock_mf.db")